### 1. Environment Setup
Here, we set a fixed random seed (`42`) across all libraries to ensure that our results stay consistent every time we run the notebook. We also check if a GPU is available to speed up the model training later.


In [1]:
import random
import numpy as np
import torch
from transformers import set_seed

def set_reproducibility(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to: {seed}")

set_reproducibility(42)

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Type: {torch.cuda.get_device_name(0)}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Note: Trained on", "GPU" if torch.cuda.is_available() else "CPU",
      "(departmental cluster recommended for faster runs if CPU used)")

Global seed set to: 42
GPU Available: True
GPU Type: Tesla T4
Note: Trained on GPU (departmental cluster recommended for faster runs if CPU used)


In [2]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas matplotlib seaborn sentencepiece

### 2. Loading the Dataset
We load the publicly available **GoEmotions** dataset using the `datasets` library. We also extract and print the 28 emotion labels to see exactly what classes our model will be predicting.


In [3]:
from datasets import load_dataset

dataset = load_dataset("go_emotions", "simplified")

label_names = dataset["train"].features["labels"].feature.names
num_labels = len(label_names)

print("Number of emotion labels:", num_labels)
print("Labels:", label_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Number of emotion labels: 28
Labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


### 3. Creating the Explicit vs. Implicit Split
Our core research question compares the model's ability to detect explicit vs. implicit emotions. We achieve this by defining an "Affective Lexicon." A text is considered **implicit** if it does not contain any obvious emotion-related words from our lexicon. This creates a new boolean column `is_implicit` in our dataset for later analysis.


In [4]:
import re

AFFECTIVE_LEXICON = set([
    'happy', 'sad', 'angry', 'fear', 'scared', 'afraid', 'surprise', 'surprised', 'disgust', 'gross', 'ew',
    'love', 'hate', 'joy', 'joyful', 'excited', 'thrilled', 'proud', 'ashamed', 'guilty', 'bored', 'lonely',
    'worried', 'nervous', 'anxious', 'calm', 'relaxed', 'grateful', 'thank', 'thanks', 'sorry', 'apologize',
    'remorse', 'regret', 'wow', 'amazing', 'awesome', 'fantastic', 'terrible', 'horrible', 'bad', 'good',
    'wonderful', 'furious', 'mad', 'pissed', 'annoyed', 'irritated', 'frustrated', 'disappointed', 'depressed',
    'sadness', 'upset', 'hurt', 'pain', 'crying', 'cry', 'laughing', 'lol', 'haha', 'funny', 'hilarious',
    'smile', 'smiling', 'frown', 'ecstatic', 'delighted', 'cheerful', 'gloomy', 'miserable', 'heartbroken',
    'enraged', 'outraged', 'terrified', 'appalled', 'revolted', 'adoring', 'envious', 'optimistic', 'pessimistic',
    'relieved', 'content', 'stunned', 'apprehensive', 'doubtful', 'indifferent', 'motivated', 'inspired',
    'discouraged', 'betrayed', 'abandoned', 'isolated', 'overwhelmed', 'exhausted', 'energetic', 'vibrant',
    'serene', 'peaceful', 'agitated', 'restless', 'grumpy', 'cranky', 'elated', 'jubilant', 'melancholy',
    'nostalgic', 'resentful', 'bitter', 'sympathetic', 'empathetic', 'compassionate', 'indignant', 'humiliated',
    'mortified', 'triumphant', 'victorious', 'defeated', 'hopeless', 'despairing', 'euphoric', 'blissful',
    'appreciative', 'thankful', 'apologetic', 'regretful', 'astonished', 'bewildered', 'perplexed', 'intrigued',
    'fascinated', 'bothered', 'disturbed', 'unsettled', 'comforted', 'reassured', 'alarmed', 'panicked',
    'horrified', 'loathing', 'repulsed', 'enchanted', 'captivated', 'enamored', 'infatuated', 'skeptical',
    'distrustful', 'confident', 'assured', 'hesitant', 'undecided', 'determined', 'resolute',
    'admiration', 'amusement', 'annoyance', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment',
    'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'love', 'nervousness',
    'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise'
])

def is_implicit(text):
    words = set(re.findall(r'\b\w+\b', text.lower()))
    return not any(word in AFFECTIVE_LEXICON for word in words)

dataset = dataset.map(lambda ex: {"is_implicit": is_implicit(ex["text"])})

train_implicit = sum(dataset['train']['is_implicit'])
print(f"Implicit in train: {train_implicit} ({train_implicit/len(dataset['train']):.2%})")

Implicit in train: 30671 (70.65%)


### 4. Data Preprocessing and Tokenization
To prepare our text for the BERT model, two steps are performed here:
1. **Multi-hot Encoding:** We convert our emotion labels into dense multi-hot vectors, a requirement for multi-label classification. Since a text can have multiple emotions, each label index is flipped to `1.0`.
2. **Tokenization:** Using the `BertTokenizer`, we map our raw strings into input IDs and setup attention masks, explicitly casting formatting to PyTorch tensors so the Trainer can consume them directly.


In [5]:
import numpy as np
from transformers import BertTokenizer

def multi_hot(ex):
    vec = np.zeros(num_labels, dtype=np.float32)
    for lbl in ex["labels"]:
        vec[lbl] = 1.0
    ex["labels"] = vec
    return ex

dataset = dataset.map(multi_hot)

from datasets import Value, Sequence
dataset = dataset.cast_column("labels", Sequence(Value("float32")))

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(ex):
    return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'], output_all_columns=True)
print("Labels dtype:", dataset['train'][0]['labels'].dtype)

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Labels dtype: torch.float32


### 5. Establishing Baselines
Before leveraging deep learning, it is an academic standard to establish performance baselines. We implement two baseline models here:
1. **Majority Dummy Classifier:** Always predicts the most common class to gauge the absolute minimum performance.
2. **TF-IDF + Logistic Regression:** A traditional machine learning pipeline that acts as our main competitor to the Transformer model. We record their Macro F1 scores to demonstrate why a robust fine-tuned setup is necessary.


In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score

train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()

X_train_text = train_df['text']
y_train = np.array(train_df['labels'].tolist())
X_val_text = val_df['text']
y_val = np.array(val_df['labels'].tolist())


majority = DummyClassifier(strategy='most_frequent')
majority.fit(X_train_text, y_train)
majority_preds = majority.predict(X_val_text)
print("Majority Macro F1:", f1_score(y_val, majority_preds, average='macro'))


tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)

lr = MultiOutputClassifier(LogisticRegression(max_iter=1000))
lr.fit(X_train_tfidf, y_train)
lr_preds = lr.predict(X_val_tfidf)
print("LR Macro F1:", f1_score(y_val, lr_preds, average='macro'))

Majority Macro F1: 0.0
LR Macro F1: 0.24759555801363176


### 6. Definitions for Comprehensive Evaluation Metrics
To keep our code modular, this cell defines reusable evaluation functions. We need to dissect the model's performance on multiple fronts: Overall Metrics (Macro/Micro F1), performance on single-label vs. multi-label texts, and a granular breakdown of predictions per emotion to understand specific strengths and failure cases.


In [7]:
from sklearn.metrics import f1_score

def compute_subset_metrics(df, pred_col, label_col, subset_name=""):
    labels = np.array(df[label_col].tolist())
    preds = np.array(df[pred_col].tolist())
    macro = f1_score(labels, preds, average='macro', zero_division=0)
    micro = f1_score(labels, preds, average='micro', zero_division=0)
    print(f"{subset_name} Macro F1: {macro:.4f} | Micro F1: {micro:.4f}")

def compute_single_multi(df, pred_col, label_col, subset_name=""):
    df['num_emotions'] = df[label_col].apply(np.sum)
    single = df[df['num_emotions'] == 1]
    multi = df[df['num_emotions'] > 1]
    s_macro = f1_score(np.array(single[label_col].tolist()), np.array(single[pred_col].tolist()), average='macro', zero_division=0) if not single.empty else 0.0
    m_macro = f1_score(np.array(multi[label_col].tolist()), np.array(multi[pred_col].tolist()), average='macro', zero_division=0) if not multi.empty else 0.0
    print(f"{subset_name} Single Macro F1: {s_macro:.4f} | Multi Macro F1: {m_macro:.4f}")

def compute_per_emotion_f1(df, pred_col, label_col, subset_name=""):
    labels = np.array(df[label_col].tolist())
    preds = np.array(df[pred_col].tolist())
    print(f"\n{subset_name} Per-Emotion F1:")
    for i, emo in enumerate(label_names):
        f1 = f1_score(labels[:, i], preds[:, i], average='binary', zero_division=0)
        print(f"{emo}: {f1:.4f}")

### 7. Evaluating the Model Before Fine-Tuning
This cell evaluates the pre-trained `bert-base-uncased` model on our validation data *before* performing any training. We calculate and print the Macro/Micro F1 scores for overall, implicit, explicit, single, and multi-emotion texts to establish our untrained baseline.


In [8]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
).to(device)

temp_args = TrainingArguments(
    output_dir="./temp",
    per_device_eval_batch_size=32,
    fp16=torch.cuda.is_available(),
)

temp_trainer = Trainer(model=model, args=temp_args)

pre_preds_output = temp_trainer.predict(dataset['validation'])
pre_preds = (torch.sigmoid(torch.tensor(pre_preds_output.predictions)) > 0.5).numpy()

val_df['pre_preds'] = list(pre_preds)

print("\n=== Before Fine-Tuning ===")
compute_subset_metrics(val_df, 'pre_preds', 'labels', "Overall")

implicit_df = val_df[val_df['is_implicit']]
explicit_df = val_df[~val_df['is_implicit']]

compute_subset_metrics(implicit_df, 'pre_preds', 'labels', "Implicit")
compute_subset_metrics(explicit_df, 'pre_preds', 'labels', "Explicit")

compute_single_multi(implicit_df, 'pre_preds', 'labels', "Implicit")
compute_single_multi(explicit_df, 'pre_preds', 'labels', "Explicit")

compute_per_emotion_f1(implicit_df, 'pre_preds', 'labels', "Implicit")
compute_per_emotion_f1(explicit_df, 'pre_preds', 'labels', "Explicit")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== Before Fine-Tuning ===
Overall Macro F1: 0.0620 | Micro F1: 0.1108
Implicit Macro F1: 0.0604 | Micro F1: 0.1122
Explicit Macro F1: 0.0596 | Micro F1: 0.1074
Implicit Single Macro F1: 0.0523 | Multi Macro F1: 0.1034


/tmp/ipython-input-7248/715275497.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['num_emotions'] = df[label_col].apply(np.sum)
/tmp/ipython-input-7248/715275497.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['num_emotions'] = df[label_col].apply(np.sum)


Explicit Single Macro F1: 0.0504 | Multi Macro F1: 0.0918

Implicit Per-Emotion F1:
admiration: 0.1301
amusement: 0.0460
anger: 0.0000
annoyance: 0.1193
approval: 0.1676
caring: 0.0000
confusion: 0.0668
curiosity: 0.1042
desire: 0.0000
disappointment: 0.0403
disapproval: 0.0785
disgust: 0.0351
embarrassment: 0.0135
excitement: 0.0252
fear: 0.0226
gratitude: 0.0155
grief: 0.0036
joy: 0.0388
love: 0.0000
nervousness: 0.0000
optimism: 0.0758
pride: 0.0000
realization: 0.0348
relief: 0.0000
remorse: 0.0000
sadness: 0.0421
surprise: 0.0417
neutral: 0.5907

Explicit Per-Emotion F1:
admiration: 0.2380
amusement: 0.2399
anger: 0.0000
annoyance: 0.0684
approval: 0.0737
caring: 0.0000
confusion: 0.0240
curiosity: 0.0462
desire: 0.0000
disappointment: 0.0000
disapproval: 0.0495
disgust: 0.0314
embarrassment: 0.0114
excitement: 0.0516
fear: 0.0569
gratitude: 0.3442
grief: 0.0060
joy: 0.0643
love: 0.0000
nervousness: 0.0000
optimism: 0.0702
pride: 0.0000
realization: 0.0274
relief: 0.0000
remorse: 

### 8. Fine-Tuning the Transformer Model
We proceed to train (fine-tune) the BERT model on our dataset for 10 epochs. We use `EarlyStopping` to halt training if the validation `macro_f1` does not improve for 3 consecutive epochs, ensuring we capture the best model without overfitting.


In [9]:
from transformers import EarlyStoppingCallback
import numpy as np
from sklearn.metrics import f1_score, precision_recall_fscore_support

def compute_metrics(p):
    predictions = (torch.sigmoid(torch.tensor(p.predictions)) > 0.5).numpy().astype(int)
    labels = p.label_ids.astype(int)


    macro_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    micro_f1 = f1_score(labels, predictions, average='micro', zero_division=0)

    metrics = {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
    }
    return metrics

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    report_to="tensorboard",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1,Micro F1
1,0.089037,0.088922,0.363210,0.549735
2,0.080228,0.083868,0.426194,0.571748
3,0.066172,0.086477,0.434706,0.576277
4,0.049308,0.097131,0.475234,0.562254
5,0.035344,0.112539,0.476335,0.567028
6,0.025006,0.123793,0.477269,0.553302
7,0.017284,0.132659,0.478272,0.566467
8,0.012057,0.142433,0.489561,0.566132
9,0.008786,0.150413,0.493278,0.561621
10,0.006475,0.152992,0.496278,0.566957


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=27140, training_loss=0.04305067575463233, metrics={'train_runtime': 3634.2407, 'train_samples_per_second': 119.447, 'train_steps_per_second': 7.468, 'total_flos': 2.85607930586112e+16, 'train_loss': 0.04305067575463233, 'epoch': 10.0})

### 9. Defining the Training Metrics Function
To monitor our model's performance correctly during training, we create a custom function passed to the `Trainer`. This function converts raw logits to binary predictions using a 0.5 threshold and calculates the Macro and Micro F1 scores for each epoch.


In [10]:
import numpy as np
from sklearn.metrics import f1_score, precision_recall_fscore_support

def compute_metrics(p):
    predictions = (torch.sigmoid(torch.tensor(p.predictions)) > 0.5).numpy().astype(int)
    labels = p.label_ids.astype(int)


    macro_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    micro_f1 = f1_score(labels, predictions, average='micro', zero_division=0)

    metrics = {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
    }
    return metrics

### 10. Evaluating the Model After Fine-Tuning
Now that the model has been fine-tuned, we generate predictions on the same validation set to assess the impact of training. We re-apply our evaluation functions to see how much the Macro/Micro F1 scores have improved for the overall dataset, as well as the explicit vs. implicit and single vs. multi-label splits.


In [11]:
post_preds_output = trainer.predict(dataset['validation'])
post_preds = (torch.sigmoid(torch.tensor(post_preds_output.predictions)) > 0.5).numpy()

val_df['post_preds'] = list(post_preds)


implicit_df = val_df[val_df['is_implicit']]
explicit_df = val_df[~val_df['is_implicit']]

print("\n=== After Fine-Tuning ===")
compute_subset_metrics(val_df, 'post_preds', 'labels', "Overall")

compute_subset_metrics(implicit_df, 'post_preds', 'labels', "Implicit")
compute_subset_metrics(explicit_df, 'post_preds', 'labels', "Explicit")

compute_single_multi(implicit_df, 'post_preds', 'labels', "Implicit")
compute_single_multi(explicit_df, 'post_preds', 'labels', "Explicit")

compute_per_emotion_f1(implicit_df, 'post_preds', 'labels', "Implicit")
compute_per_emotion_f1(explicit_df, 'post_preds', 'labels', "Explicit")


=== After Fine-Tuning ===
Overall Macro F1: 0.4963 | Micro F1: 0.5670
Implicit Macro F1: 0.4321 | Micro F1: 0.5325
Explicit Macro F1: 0.4858 | Micro F1: 0.6459
Implicit Single Macro F1: 0.4528 | Multi Macro F1: 0.3883
Explicit Single Macro F1: 0.4799 | Multi Macro F1: 0.4833

Implicit Per-Emotion F1:
admiration: 0.7002
amusement: 0.6328
anger: 0.4429
annoyance: 0.3534
approval: 0.3408
caring: 0.4395
confusion: 0.4195
curiosity: 0.4891
desire: 0.5303
disappointment: 0.2707
disapproval: 0.3774
disgust: 0.4557
embarrassment: 0.4681
excitement: 0.2800
fear: 0.5676
gratitude: 0.4194
grief: 0.5000


/tmp/ipython-input-7248/715275497.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['num_emotions'] = df[label_col].apply(np.sum)
/tmp/ipython-input-7248/715275497.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['num_emotions'] = df[label_col].apply(np.sum)


joy: 0.5275
love: 0.6241
nervousness: 0.0000
optimism: 0.5994
pride: 0.2500
realization: 0.2749
relief: 0.2105
remorse: 0.2667
sadness: 0.4531
surprise: 0.5426
neutral: 0.6621

Explicit Per-Emotion F1:
admiration: 0.7332
amusement: 0.8235
anger: 0.5412
annoyance: 0.2689
approval: 0.3378
caring: 0.3284
confusion: 0.3784
curiosity: 0.4286
desire: 0.3571
disappointment: 0.2632
disapproval: 0.3434
disgust: 0.2857
embarrassment: 0.4444
excitement: 0.3678
fear: 0.6905
gratitude: 0.9496
grief: 0.0000
joy: 0.6429
love: 0.8462
nervousness: 0.6667
optimism: 0.5210
pride: 0.8000
realization: 0.2500
relief: 0.2000
remorse: 0.6723
sadness: 0.5802
surprise: 0.5652
neutral: 0.3170


### 11. Identifying the Hardest Implicit Emotions
To better understand the model's limitations, we isolate the implicit cases and calculate the specific F1 scores for each of the 28 emotions. Sorting these scores allows us to pinpoint which emotion categories the model struggles with the most when contextual cues are subtle.


In [12]:
implicit_val = val_df[val_df['is_implicit']]

per_emotion_f1 = {}
for i, emo in enumerate(label_names):
    f1 = f1_score(
        implicit_val['labels'].apply(lambda x: x[i]),
        implicit_val['post_preds'].apply(lambda x: x[i]),
        average='binary',
        zero_division=0
    )
    per_emotion_f1[emo] = f1

sorted_difficult = sorted(per_emotion_f1.items(), key=lambda x: x[1])

print("Most difficult implicit emotions (lowest F1 after fine-tuning):")
for emo, score in sorted_difficult[:10]:
    print(f"{emo}: {score:.4f}")

Most difficult implicit emotions (lowest F1 after fine-tuning):
nervousness: 0.0000
relief: 0.2105
pride: 0.2500
remorse: 0.2667
disappointment: 0.2707
realization: 0.2749
excitement: 0.2800
approval: 0.3408
annoyance: 0.3534
disapproval: 0.3774


### 12. Qualitative Error Analysis
Looking beyond aggregate metrics, we perform a qualitative review of actual model mistakes. By isolating cases where the model prediction does not perfectly match the true labels, and printing 10 samples, we can manually inspect context misinterpretations and understand where natural language ambiguity tricked the model.


In [13]:
errors = val_df[~val_df.apply(lambda row: np.array_equal(row['labels'], row['post_preds']), axis=1)]

print("\nError Analysis (first 10 misclassified samples after training):")
for i in range(min(10, len(errors))):
    print(f"\nText: {errors.iloc[i]['text']}")
    true_labels = [label_names[j] for j in np.where(errors.iloc[i]['labels'] > 0.5)[0]]
    pred_labels = [label_names[j] for j in np.where(errors.iloc[i]['post_preds'] > 0.5)[0]]
    print(f"True: {true_labels}")
    print(f"Pred: {pred_labels}")


Error Analysis (first 10 misclassified samples after training):

Text: Is this in New Orleans?? I really feel like this is New Orleans.
True: ['neutral']
Pred: ['confusion', 'curiosity']

Text: You know the answer man, you are programmed to capture those codes they send you, don’t avoid them!
True: ['approval', 'neutral']
Pred: ['neutral']

Text: The economy is heavily controlled and subsidized by the government. In any case, I was poking at the lack of nuance in US politics today
True: ['approval', 'neutral']
Pred: ['neutral']

Text: He could have easily taken a real camera from a legitimate source and change the price in Word/Photoshop and then print it out.
True: ['optimism']
Pred: ['neutral']

Text: Wah Mum other people call me on my bullshit and I can't ban them , Go out side son.
True: ['anger']
Pred: ['annoyance']

Text: There it is!
True: ['neutral']
Pred: ['excitement']

Text: At least now [NAME] has more time to gain his confidence
True: ['optimism']
Pred: ['neutral']

Text:

### 13. Comprehensive Classification Report
As a final evaluation step, we generate a full `classification_report`. This provides an industry-standard, detailed breakdown of Precision, Recall, and F1-score for all 28 emotions across the entire validation set. We use this tabular view to concisely summarize the model's discriminative power per class.


In [14]:
from sklearn.metrics import classification_report

print("\nFull Per-Emotion Classification Report (After Fine-Tuning):")
print(classification_report(
    np.stack(val_df['labels'].values),
    np.stack(val_df['post_preds'].values),
    target_names=label_names,
    zero_division=0
))


Full Per-Emotion Classification Report (After Fine-Tuning):
                precision    recall  f1-score   support

    admiration       0.69      0.74      0.71       488
     amusement       0.74      0.81      0.77       303
         anger       0.49      0.45      0.47       195
     annoyance       0.30      0.38      0.34       303
      approval       0.35      0.33      0.34       397
        caring       0.44      0.39      0.41       153
     confusion       0.41      0.41      0.41       152
     curiosity       0.44      0.52      0.48       248
        desire       0.48      0.52      0.50        77
disappointment       0.29      0.25      0.27       163
   disapproval       0.39      0.36      0.37       292
       disgust       0.36      0.46      0.41        97
 embarrassment       0.50      0.43      0.46        35
    excitement       0.33      0.31      0.32        96
          fear       0.74      0.56      0.63        90
     gratitude       0.90      0.91      0